In [119]:
!pip install segmentation-models-pytorch

In [1]:
import os
import pandas as pd
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader
import torchvision.models as models
import segmentation_models_pytorch as smp
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

Analyse of the dataset

In [2]:
# Load the file containing images labels

label_train = pd.read_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/train/Quality_Assessment_train.csv', sep=';')
label_test = pd.read_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/test/Quality_Assessment_test.csv', sep=';')


In [ ]:
# Change the order of columns of label_train and label_test
new_order = ['Number', 'Disease', 'IC', 'Blur', 'LC']

mapping = {'A': 0, 'D': 1, 'G': 2, 'N': 3}
reverse = {0: 'A', 1: 'D', 2: 'G', 3: 'N'}

label_train = label_train[new_order]
label_train['Disease'] = label_train['Disease'].map(mapping)


label_train.to_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/train/Quality_Assessment_train_clean.csv', index=False, sep=';')

label_test = label_test[new_order]
label_test['Disease'] = label_test['Disease'].map(mapping)
label_test.to_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/test/Quality_Assessment_test_clean.csv', index=False, sep=';')

In [7]:
label_train = pd.read_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/train/Quality_Assessment_train_clean.csv', sep=';')
label_test = pd.read_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/test/Quality_Assessment_test_clean.csv', sep=';')

In [8]:
train_size = len(label_train)
test_size = len(label_test)

print("Repartition % of diseases in train and test set :")
print(label_train['Disease'].value_counts() *100 / train_size )
print(label_test['Disease'].value_counts() *100 / test_size )

Repartition % of diseases in train and test set :
Disease
0    25.0
1    25.0
2    25.0
3    25.0
Name: count, dtype: float64
Disease
0    25.0
1    25.0
2    25.0
3    25.0
Name: count, dtype: float64


In [9]:
print("Repartition % of Illumination Color in train and test set :")
print(label_train['IC'].value_counts() *100 / train_size )
print(label_test['IC'].value_counts() *100 / test_size )


print("Repartition % of Blur in train and test set :")
print(label_train['Blur'].value_counts() *100 / train_size )
print(label_test['Blur'].value_counts() *100 / test_size )


print("Repartition % of Low Contrast in train and test set :")
print(label_train['LC'].value_counts() *100 / train_size )
print(label_test['LC'].value_counts() *100 / test_size )


Repartition % of Illumination Color in train and test set :
IC
1    82.666667
0    17.333333
Name: count, dtype: float64
IC
1    74.0
0    26.0
Name: count, dtype: float64
Repartition % of Blur in train and test set :
Blur
1    85.0
0    15.0
Name: count, dtype: float64
Blur
1    79.5
0    20.5
Name: count, dtype: float64
Repartition % of Low Contrast in train and test set :
LC
1    96.666667
0     3.333333
Name: count, dtype: float64
LC
1    92.5
0     7.5
Name: count, dtype: float64


#### Data augmentation

In [ ]:
"""def show_img(path_img, title = None):
  
  I = Image.open(path_img)
  Inp = np.array(I)
  plt.imshow(I)
  plt.title(title)
  plt.axis('off')
  plt.show()"""

Try to blur some images to see the result.

In [ ]:
"""not_blurred_imgs = label_train[label_train["Blur"]==1].head(5)
#print(not_blurred_imgs)
blurred_imgs = label_train[(label_train["Blur"]==0) & (label_train["LC"]==0)]
print(f"Number of blurred images : {len(blurred_imgs)}")
print(blurred_imgs.head(10))

path_img_not_blurred = "/content/gdrive/MyDrive/Colab Notebooks/computer_vision_and_deep_learning/dataset_project/train/imgs/1_A.png"
show_img(path_img_not_blurred, "Original image")
I = cv2.imread(path_img_not_blurred)

I_blurred = cv2.GaussianBlur(I, (45,45), 0)
I_blurred = cv2.cvtColor(I_blurred, cv2.COLOR_BGR2RGB)
plt.imshow(I_blurred)
plt.title("Blurred version 1")
plt.axis('off')
plt.show()

I_blurred_2 = cv2.GaussianBlur(I, (55,55), 0)
I_blurred_2 = cv2.cvtColor(I_blurred_2, cv2.COLOR_BGR2RGB)
plt.imshow(I_blurred_2)
plt.title("Blurred version 2")
plt.axis('off')
plt.show()

I_blurred_3 = cv2.GaussianBlur(I, (61,61), 0)
I_blurred_3 = cv2.cvtColor(I_blurred_3, cv2.COLOR_BGR2RGB)
plt.imshow(I_blurred_3)
plt.title("Blurred version 3")
plt.axis('off')
plt.show()

path_img_blurred = "/content/gdrive/MyDrive/Colab Notebooks/computer_vision_and_deep_learning/dataset_project/train/imgs/44_A.png"
show_img(path_img_blurred, "A blurred original image")"""

In [ ]:
def get_images(path_folder_img, path_folder_mask, df_labels):
  """ This function retrieves high-quality images 
  Args : 
  path_folder_img : path of training images
  path_folder_mask : path of training masks
  df_labels : data frame containing information about image class

  Returns :
  list_path_imgs : list containing paths of high-quality images
  list_path_masks : list containing the associated masks
  """
  
  list_path_imgs = []
  
  good_imgs = df_labels[(df_labels["Blur"] == 1) & (df_labels["LC"] == 1) & (df_labels["IC"] == 1)]
  
  good_imgs = good_imgs.sample(frac = 0.2, random_state = 42)
  list_path_masks = []
  

  for _,line in good_imgs.iterrows():
    nb_good_img = line["Number"]
    disease = line["Disease"]
    conversion = {0: 'A', 1: 'D', 2: 'G', 3: 'N'}
    disease = conversion[disease]
    path_good_img = f"{path_folder_img}/{nb_good_img}_{disease}.png"
    
    path_mask_good_img = f"{path_folder_mask}/{nb_good_img}_{disease}.png"
    list_path_imgs.append(path_good_img)
    list_path_masks.append(path_mask_good_img)
    
  return list_path_imgs, list_path_masks 




def data_augmentation(list_path_img, list_path_masks, df_labels):
    """ 
    Apply data augmentation on images by flipping them.
    """

    count_imgs = 600
    lines = []

    for i in range(len(list_path_img)):###########################
      path = list_path_img[i]#####################
      mask_path = list_path_masks[i]
      count_imgs += 1
      name_img = Path(path).stem #delete .png
      idx_img = int(name_img[:-2])

      line = df_labels[df_labels["Number"] == idx_img]
      disease = line.iloc[0,1]
      lines.append({"Number": count_imgs, "Disease": disease, "IC": line.iloc[0,1], "Blur": line.iloc[0,2], "LC" : line.iloc[0,3] })
      conversion = {0: 'A', 1: 'D', 2: 'G', 3: 'N'}
      disease = conversion[disease]

      I = cv2.imread(path)
      I_flip = cv2.flip(I, 1) #horizontal flip

      M = cv2.imread(mask_path)###########################
      M = cv2.flip(M, 1)###############################

      flip_img_path = f"{os.path.dirname(path)}/{count_imgs}_{disease}.png"
      mask_flip_img_path = f"{os.path.dirname(mask_path)}/{count_imgs}_{disease}.png" ###########################
      cv2.imwrite(flip_img_path, I_flip)
      cv2.imwrite(mask_flip_img_path, M)################################
      #print(blurred_img_path)



    lines = pd.DataFrame(lines)
    df_labels_increased = pd.concat([df_labels, lines], ignore_index = True)
    df_labels_increased.to_csv("/content/gdrive/MyDrive/Colab Notebooks/computer_vision_and_deep_learning/dataset_project/Quality Assessment.xlsx - Train_clean.csv", index = False)
    print(f"Data augmentation done.")

#### Training

In [11]:
class CustomImageDataset(Dataset):
    """ Create a personalized Dataset class."""

    def __init__(self, annotations_file, img_dir, mask_dir, img_transform=None, mask_transform=None):
        """ Set up paths and images transformations.

        Args :
        annotation_file : pandas data frame containing images labels.
        img_dir : path of the directory where images are stored.
        mask_dir : path of the directory where masks are stored.
        img_transform : transformations that will be applied to images.
        mask_transform : transformations that will be applied to masks. """

        self.img_labels = annotations_file 
        self.img_dir = img_dir
        self.mask_dir = mask_dir 
        self.img_transform = img_transform
        self.mask_transform = mask_transform

    def __len__(self):
        """ Return the number of samples in the dataset."""

        return len(self.img_labels)
   
    def __getitem__(self, idx):
        """ For a given index, it returns the sample (image, mask, label) associated to it.
        Args : 
        idx : index of a sample.

        Returns:
        image : tensor of shape [C,H,W].
        mask : tensor of shape [1, H, W].
        label : tensor of shape [1,4] containing information about the image.
        """
        
        # Reconstruction of the image's name
        name_image = str(self.img_labels.iloc[idx, 0]) + '_' + reverse[int(self.img_labels.iloc[idx, 1])]+ '.png'
        
        # Reconstruction of the image's path
        img_path = os.path.join(self.img_dir, name_image)
        # Reconstruction of the mask's path
        mask_path = os.path.join(self.mask_dir, name_image)
        
        # Open the image so it can be transformed
        image = Image.open(img_path).convert('RGB') #some images have 4 channels 
        # Open the mask so it can be transformed
        mask = Image.open(mask_path).convert('L')
        # Retrieve information about the image
        label = torch.tensor(self.img_labels.iloc[idx, 1:].values.astype('float32'), dtype=torch.float32)
        
        # Apply transformations on the image
        if self.img_transform:
            image = self.img_transform(image)

        # Apply transformations on the mask
        if self.mask_transform:
            mask = self.mask_transform(mask)
   
        return image, mask, label
    

In [12]:

# List of transformations that will be applied to images and masks
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor() 
])

path_dataset ="/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset"

fundus_dataset_train = CustomImageDataset(annotations_file= label_train, img_dir = path_dataset + "/train/imgs", mask_dir = path_dataset + "/train/masks", img_transform=transform, mask_transform=transform)
fundus_dataset_test = CustomImageDataset(annotations_file= label_test, img_dir = path_dataset + "/test/imgs", mask_dir = path_dataset + "/test/masks", img_transform=transform, mask_transform=transform)

train_dataloader = DataLoader(fundus_dataset_train, batch_size=64, shuffle=True)
test_dataloader= DataLoader(fundus_dataset_test, batch_size=64, shuffle=True)

In [ ]:
def train_loop_basic(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta):

    size = len(dataloader.dataset)  
    model = model.to(device)
    model.train()

    # This list will contain lists. Each of them will contain an image, the associated mask (ground truth) and the predicted mask
    list_imgs_each_epoch = []

    k = 0
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_loss_class = 0.0
        epoch_loss_seg = 0.0
        
        # This dictionary will contain the number of predictions for each class ({A, D, G, N}_tot)
        # and the number of good predictions done for each class ({A, D, G, N}_true)
        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
        
        # This list will contain the image, the mask (ground truth) and the predicted mask of the first image of the first batch of the current epoch
        L = []
        

        for image, mask, label in dataloader:
            
            # Put images, masks and labels of the batch in the same device
            image= image.to(device)
            mask= mask.to(device)
            label= label.to(device)
           
            
            # Set gradient to zero
            optimizer.zero_grad()

            # Apply the model to each image of the batch.
            # pred_segm : tensor of shape [B,1,H,W]
            # B : batch size, 1: number of channels, H: height of the mask, W: width of the mask
            # pred_segm contains raw logits
            # pred_class : tensor of shape [B, N]
            # N : number of classes
            # pred_class contains raw logits
            pred_segm, pred_class = model(image)


            # For each image, chose the class with the higher score
            pred_class_indices = pred_class.argmax(dim=1)
            
            batch_loss_class = 0
            batch_loss_seg = 0

            # Count the number of predictions and the number of good prediction for each class
            for i in range (len(label)):
                label_name = reverse[label[i,0].item()]
                dic_acc_disease[label_name + "_tot"] += 1 
                if label[i,0].item() == pred_class_indices[i].item() :
                   dic_acc_disease[label_name + "_true"] += 1
            

            batch_loss_seg = loss_seg(pred_segm, mask)
            batch_loss_class = loss_class(pred_class, label[:,0].long())
                    
            # Apply a penalty alpha for the loss associated to the classification and betafor the one associated to segmentation
            batch_loss_class = alpha * batch_loss_class
            epoch_loss_class += batch_loss_class.item()
            batch_loss_seg = beta * batch_loss_seg
            epoch_loss_seg += batch_loss_seg.item()
           
            batch_loss = (batch_loss_class + batch_loss_seg) / len(label) 
            
            # Backpropagation
            batch_loss.backward()
            optimizer.step()
        
            # Retrieve the first image of the first batch and its mask 
            if k == 1: # We dont't want to store anything of the first batch of the first epoch because they are not representative of the model's performance
                L.append(image[0].detach().cpu().permute(1,2,0).numpy())   # check
                L.append(mask[0].detach().cpu().permute(1,2,0).numpy())
                L.append(pred_segm[0].detach().cpu().permute(1,2,0).numpy())
                k = 2

        epoch_loss_class /= size
        epoch_loss_seg /= size
    
        epoch_loss = epoch_loss_class + epoch_loss_seg


        list_imgs_each_epoch.append(L)
        
        k = 1 
        
        print(f'Epoch {epoch+1}/{num_epochs}, Total Loss: {epoch_loss},Classification Loss: {epoch_loss_class}, Total Segmentation Loss: {epoch_loss_seg},Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')

    return list_imgs_each_epoch
    


In [14]:
def test_loop_basic(dataloader, model, loss_seg, loss_class, device, alpha, beta):

    size = len(dataloader.dataset)  
    model = model.to(device)
    model.eval()

    
    test_loss = 0.0
    test_loss_seg = 0.0
    test_loss_class = 0.0

    
    # This dictionary will contain the number of predictions for each class ({A, D, G, N}_tot)
    # and the number of good predictions made for each class ({A, D, G, N}_true)
    dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
    
    # Store predicted labels for the confusion matrix
    all_pred = []
    # Store true labels for the confusion matrix
    all_labels = []

    with torch.no_grad():
        for image, mask, label in dataloader:
            
            # Put images, masks and labels of the current batch in the same device
            image= image.to(device)
            mask= mask.to(device)
            label= label.to(device)
        
            # Apply the model on each image of the batch
            # pred_segm : tensor of shape [B, 1, H, W]. It contains raw logits.
            # pred_clss : tensor of shape [B, N]. It contains raw logits
            pred_segm, pred_class = model(image)
            # For each image, chose the class that has the highest score
            pred_class_indices = pred_class.argmax(dim=1)

            
            # Store the predicted label of each image of the batch
            all_pred.extend(pred_class_indices.cpu().numpy())
            # Store the true label of each image of the batch
            all_labels.extend(label[:,0].cpu().numpy())

            # Count the number of predictions and the number of good predictions made for each class
            for i in range (len(label)):
                label_name = reverse[label[i,0].item()]
                dic_acc_disease[label_name + "_tot"] += 1 
                if label[i,0].item() == pred_class_indices[i].item() :
                   dic_acc_disease[label_name + "_true"] += 1
            
            # Apply a penalty to the loss segmentation and to the loss classification
            test_loss_seg += beta * loss_seg(pred_segm, mask)
            test_loss_class += alpha * loss_class(pred_class, label[:,0].long())
                    
        
        test_loss_seg /= size
        test_loss_class /= size
        test_loss = test_loss_class + test_loss_seg

        
        
        print(f' Total Loss: {test_loss},Classification Loss: {test_loss_class}, Total Segmentation Loss: {test_loss_seg},Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')
    return all_pred, all_labels

This is our first implementation which is not the best as we are calculating the loss for each image of a batch and not for the batch. In a second time we propose the train_loop_faster which uses a mask to solve this problem. 

In [ ]:
def train_loop(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama):

    size = len(dataloader.dataset)  
    model = model.to(device)
    model.train()

    # This list will contain the image, the mask (ground truth) and the predicted mask of the first image of the batch at each epoch
    list_imgs_each_epoch = []
    
    
    k = 0  # These variables are defined to retrieve the first image of the first batch at each epoch but not for the first epoch because the model is not trained yet and the predictions are not representative of the model's performance
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_loss_class = 0.0
        epoch_loss_seg = 0.0
        epoch_loss_seg_blur = 0.0


        # This dictionary will contain the number of predictions for each class ({A, D, G, N}_tot)
        # and the number of good predictions made for each class ({A, D, G, N}_true)
        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
        
        L = [] # This list will contain the image, the mask (ground truth) and the predicted mask of the first image of the first batch of the current epoch
        
        

        for image, mask, label in dataloader:
        
            image= image.to(device)
            mask= mask.to(device)
            label= label.to(device)

            # Set gradient to zero
            optimizer.zero_grad()

            # Apply the model to each image of the batch
            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
            batch_loss_seg_blur = 0
            n_blurred = 0 # Count the number of blurred images in the batch
            
        
            for i in range (len(label)):
                # Count the number of predictions and the number of good predictions made for each class
                label_name = reverse[label[i,0].item()]
                dic_acc_disease[label_name + "_tot"] += 1 
                if label[i,0].item() == pred_class[i].argmax(dim=0).item() :
                   dic_acc_disease[label_name + "_true"] += 1

                # Apply penalty to blurred images and those with low contrast
                loss_classi = loss_class(pred_class[i].unsqueeze(0), label[i,0].unsqueeze(0).long()) 
                loss_segi = loss_seg(pred_segm[i].unsqueeze(0), mask[i].unsqueeze(0))  # unsqueeze
                if ((label[i,3].item() == 0) or (label[i,2].item() == 0)): # we want to be able to detecte cataract disease index 3 for LC and index 2 for Blur
                    batch_loss_seg +=  gama * loss_segi
                    batch_loss_seg_blur += loss_segi
                    batch_loss_class += gama * loss_classi   
                    n_blurred += 1
                else : 
                    batch_loss_seg +=  loss_segi
                    batch_loss_class += loss_classi
                    
            

            if k == 1: # We dont't want to store anything of the first batch of the first epoch because they are not representative of the model's performance
                L.append(image[0].detach().cpu().permute(1,2,0).numpy())   # check
                L.append(mask[0].detach().cpu().permute(1,2,0).numpy())
                L.append(pred_segm[0].detach().cpu().permute(1,2,0).numpy())
                k = 2
                    
            
            batch_loss_class = alpha * batch_loss_class
            epoch_loss_class += batch_loss_class.item()
            batch_loss_seg = beta * batch_loss_seg
            epoch_loss_seg += batch_loss_seg.item()
            epoch_loss_seg_blur += batch_loss_seg_blur.item() / n_blurred if n_blurred > 0 else 0
            batch_loss = (batch_loss_class + batch_loss_seg) / len(label) 
            
            # Backpropagation
            batch_loss.backward()
            optimizer.step()
        
    
        epoch_loss_class /= size
        epoch_loss_seg /= size
        epoch_loss = epoch_loss_class + epoch_loss_seg
        
        k = 1
        
        list_imgs_each_epoch.append(L)

      
        print(f'Epoch {epoch+1}/{num_epochs}, Total Loss: {epoch_loss},Classification Loss: {epoch_loss_class}, Segmentation Loss: {epoch_loss_seg}, Segmentation Loss blurred : {epoch_loss_seg_blur}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')

    return list_imgs_each_epoch



    

In [17]:
def test_loop(dataloader, model, loss_seg, loss_class, device, alpha, beta, gama):
    size = len(dataloader.dataset)
    
    model = model.to(device)
    model.eval()
    test_loss = 0.0
    test_loss_seg = 0.0
    test_loss_seg_blur = 0.0
    test_loss_class = 0.0
    n_blurred = 0
    dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
    
    with torch.no_grad():
        for image, mask, label in dataloader:
            image = image.to(device)
            mask = mask.to(device)
            label = label.to(device)

            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
            batch_loss_seg_blur = 0
             
            for i in range (len(label)):
                label_name = reverse[label[i,0].item()]
                dic_acc_disease[label_name + "_tot"] += 1 
                if label[i,0].item() == pred_class[i].argmax(dim=0).item() :
                   dic_acc_disease[label_name + "_true"] += 1
             
                loss_classi = loss_class(pred_class[i].unsqueeze(0), label[i,0].unsqueeze(0).long())   # unsqueeze
                loss_segi = loss_seg(pred_segm[i].unsqueeze(0), mask[i].unsqueeze(0))
                if ((label[i,3].item() == 0) or (label[i,2].item() == 0)): # we want to be able to detecte cataract disease index 3 for LC and index 2 for Blur
                    batch_loss_seg +=  gama * loss_segi
                    batch_loss_class += gama * loss_classi   
                    batch_loss_seg_blur += loss_segi
                    n_blurred += 1
                else : 
                    batch_loss_seg +=  loss_segi
                    batch_loss_class += loss_classi
                    
            test_loss_class += alpha * batch_loss_class.item()        
            test_loss_seg += beta * batch_loss_seg.item()
            test_loss_seg_blur += batch_loss_seg_blur.item() 
             
                   

        test_loss_seg /= size
        test_loss_class /= size
        test_loss = test_loss_class + test_loss_seg
        test_loss_seg_blur /= n_blurred if n_blurred > 0 else 1
            
    print(f'Total Loss: {test_loss},Classification Loss: {test_loss_class}, Segmentation Loss: {test_loss_seg}, Segmentation Loss Blurred : {test_loss_seg_blur},Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')

try to not see image by image in a batch in order to gain time 

In [ ]:
def train_loop_faster(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama):

    size = len(dataloader.dataset)  
    model = model.to(device)
    model.train()
    
    # This list will contain the image, the mask (ground truth) and the predicted mask
    # of the first image of the batch at each epoch
    list_imgs_each_epoch = []
    
    k = 0  # These variables are defined to retrieve the first image of the first batch at each epoch but not for the first epoch because the model is not trained yet and the predictions are not representative of the model's performance
    
    for epoch in range(num_epochs):
        epoch_loss_class = 0.0
        epoch_loss_seg = 0.0
        epoch_loss_seg_blur = 0.0

        # This dictionary will contain the number of predictions for each class ({A, D, G, N}_tot)
        # and the number of good predictions made for each class ({A, D, G, N}_true)
        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
        
        L = [] # This list will contain the image, the mask (ground truth) and the predicted mask of the first image of the first batch of the current epoch


        for image, mask, label in dataloader:
            
            # Put images, masks and labels of the batch in the same device
            image= image.to(device)
            mask= mask.to(device)
            label= label.to(device)


            # Set gradient to zero
            optimizer.zero_grad()

            # Apply the model to each image present in the batch
            # pred_segm : tensor of shape [B, 1, H, W]
            # B: number of images in the batch, 1 channel, H: height of the predicted mask, W: width of the predicted mask
            # pred_segm contains raw logits
            # pred_class : tensor of shape [B, N]
            # N: total number of classes
            #pred_class contains raw logits
            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
            
            # Apply classification loss function
            loss_c= loss_class(pred_class, label[:,0].long()) 
            # Apply segmentation loss function
            loss_s = loss_seg(pred_segm, mask)  
            
            # Tensor containing booleans : True if the image is blurred or has low contrast, False otherwise
            penalty = (label[:, 2] == 0) | (label[:, 3] == 0)
            
            # Tensor that enables setting a penalty (weigh = gama) if the image is blurred or has low contrast (otherwise weight  = 1) 
            weights = torch.where(penalty, torch.tensor(gama, dtype=torch.float32, device=device), torch.tensor(1.0, dtype=torch.float32, device=device))

            # Compute the classification loss and the segmentation loss considering the penalty of blurred images 
            # and those with low contrast
            batch_loss_class = (loss_c * weights).sum()
            batch_loss_seg = (loss_s.mean(dim=[1, 2, 3]) * weights).sum()  #######
            
            # Apply a penalty to the classification loss and to the segmentation loss
            batch_loss_class = alpha * batch_loss_class
            epoch_loss_class += batch_loss_class.item()
            batch_loss_seg = beta * batch_loss_seg
            # Total loss of a batch
            batch_loss = (batch_loss_class + batch_loss_seg ) / len(label)
            
            epoch_loss_seg += batch_loss_seg.item()
            
            # For each image, chose the class with the highest score
            pred_label = pred_class.argmax(dim=1)

            # Count the number of predictions and the number of good predictions for each class
            for i in range (len(dic_acc_disease) // 2):
                disease = reverse[i]
                idx_pred = (pred_label == i)
                idx_true = (label[:, 0] == i)
                
                dic_acc_disease[disease + "_tot"] += idx_true.sum().item()
                dic_acc_disease[disease + "_true"] += (idx_pred & idx_true).sum().item()
                

            if k == 1: # We dont't want to store anything of the first batch of the first epoch because they are not representative of the model's performance
                L.append(image[0].detach().cpu().permute(1,2,0).numpy())   # check
                L.append(mask[0].detach().cpu().permute(1,2,0).numpy())
                L.append(pred_segm[0].detach().cpu().permute(1,2,0).numpy())
                k = 2
                           
            # Backpropagation
            batch_loss.backward()
            optimizer.step()

        
        epoch_loss_class /= size
        epoch_loss_seg /= size
        epoch_loss = epoch_loss_class + epoch_loss_seg

        k = 1
        list_imgs_each_epoch.append(L)
    
        
        print(f'Epoch {epoch+1}/{num_epochs}, Total Loss: {epoch_loss},Classification Loss: {epoch_loss_class}, Segmentation Loss: {epoch_loss_seg}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')


    return list_imgs_each_epoch
    


In [ ]:
def decode_segmap(img):
  """ This function displays the segmentation mask.
  Args : 
  img : numpy array containing ones (for the optic nerve) and zeros (for the background)
  """
  label_colors = np.array([(0,0,0), (255,255,255)])

  r = np.zeros_like(img).astype(np.uint8)
  g = np.zeros_like(img).astype(np.uint8)
  b = np.zeros_like(img).astype(np.uint8)

  for i in range(2):
    idx = img == i
    r[idx] = label_colors[i,0]
    g[idx] = label_colors[i,1]
    b[idx] = label_colors[i,2]

  return np.stack([r,g,b], axis = 2)



def imgs_per_epoch(list_imgs):
  """ This function displays images contained in the input list.
  Args :
  list_imgs : list containing an original image (numpy array), the corresponding mask - ground truth (numpy array)
  and the predicted mask (containing raw logits)
  """
  count = 0
  for l in list_imgs:
    count += 1

    print(f"Epoch {count}")

    img = l[0]
    plt.imshow(img)
    plt.title("Fundus")
    plt.axis("off")
    plt.show()

    mask = l[1]
    plt.imshow(mask, cmap = "gray", vmin = 0, vmax = 1)
    plt.title("Mask - ground truth")
    plt.axis("off")
    plt.show()

    predicted_mask = l[2].squeeze()
    #print(predicted_mask.shape)
    predicted_mask = torch.sigmoid(predicted_mask)
    predicted_mask = ( predicted_mask > 0.5).int()
    Inp_mask_pred = decode_segmap(predicted_mask.cpu().squeeze(0).numpy())
    plt.imshow(Inp_mask_pred)
    plt.title("Mask - prediction")
    plt.axis("off")
    plt.show()

In [ ]:
def test_loop_faster(dataloader, model, loss_seg, loss_class, device, alpha, beta, gama):
    size = len(dataloader.dataset)
    
    model = model.to(device)
    model.eval()
    test_loss = 0.0
    test_loss_seg = 0.0
    test_loss_class = 0.0
    # This dictionary will contain the number of predictions for each class ({A, D, G, N}_tot)
    # and the number of good predictions made for each class ({A, D, G, N}_true)
    dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
    
    # Store predicted labels for the confusion matrix
    all_pred = []
    # Store true labels for the confusion matrix
    all_labels = []

    with torch.no_grad():
        for image, mask, label in dataloader:
            # Put images, masks and labels of the batch in the same device as the model
            image = image.to(device)
            mask = mask.to(device)
            label = label.to(device)

            # Apply the model to each image of the batch
            # pred_segm : tensor with shape [B, 1, H, W]
            # It contains raw logits.
            # pred_class : tensor with shape [B,N]
            # It contains raw logits.
            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
            
            # Compute the classification loss
            loss_c= loss_class(pred_class, label[:,0].long()) 
            # Compute the segmentation loss
            loss_s = loss_seg(pred_segm, mask)  # unsqueeze
            
            # Tensor containing booleans : True if the image is blurred or has low contrast, False otherwise
            penalty = (label[:, 2] == 0) | (label[:, 3] == 0)
            # Tensor that enables setting a penalty (weigh = gama) if the image is blurred or has low contrast (otherwise weight  = 1) 
            weights = torch.where(penalty, torch.tensor(gama, dtype=torch.float32, device=device), torch.tensor(1.0, dtype=torch.float32, device=device))
            
            # Compute the classification loss and the segmentation loss considering the penalty of blurred images 
            # and those with low contrast
            batch_loss_class = (loss_c * weights).sum()
            batch_loss_seg = (loss_s.mean(dim=[1, 2, 3]) * weights).sum()  #######
            # Apply a penalty to the segmentation loss and the classification loss
            batch_loss_class = alpha * batch_loss_class
            batch_loss_seg = beta * batch_loss_seg
            
            test_loss_seg += batch_loss_seg.item()
            test_loss_class += batch_loss_class.item()
            
            # For each image, chose the class with the highest score
            pred_label = pred_class.argmax(dim=1)
            # Count the number of predictions and the number of good predictions made for each class
            
            # Store the predicted label of each image of the batch
            all_pred.extend(pred_label.cpu().numpy())
            # Store the true label of each image of the batch
            all_labels.extend(label[:,0].cpu().numpy())

            for i in range (len(dic_acc_disease) // 2):
                disease = reverse[i]
                idx_pred = (pred_label == i)
                idx_true = (label[:, 0] == i)
                
                dic_acc_disease[disease + "_tot"] += idx_true.sum().item()
                dic_acc_disease[disease + "_true"] += (idx_pred & idx_true).sum().item()
                
            
        test_loss_seg /= size
        test_loss_class /= size  
        test_loss = test_loss_class + test_loss_seg
        
            
    print(f'Total Loss: {test_loss},Classification Loss: {test_loss_class}, Segmentation Loss: {test_loss_seg}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')
    return all_pred, all_labels

In [31]:
'''
def train_loop_faster2(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama):

    size = len(dataloader.dataset)  
    model = model.to(device)
    model.train()
    
    
    for epoch in range(num_epochs):
        epoch_loss_class = 0.0
        epoch_loss_seg = 0.0
        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
        for image, mask, label in dataloader:
        
            image= image.to(device)
            mask= mask.to(device)
            label= label.to(device)
            
            optimizer.zero_grad()
        
            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
             
            loss_c= loss_class(pred_class, label[:,0].long()) 
            loss_s = loss_seg(pred_segm, mask)  # unsqueeze
            
            penalty = (label[:, 2] == 0) | (label[:, 3] == 0)
            
            weights = torch.where(penalty, torch.tensor(gama, dtype=torch.float32, device=device), torch.tensor(1.0, dtype=torch.float32, device=device))
            weights_seg = weights.view(-1, 1, 1, 1)
            
            batch_loss_class = (loss_c * weights).sum()
            batch_loss_seg = loss_seg(pred_segm * weights_seg, mask * weights_seg) * image.size(0)
            batch_loss_class = alpha * batch_loss_class
            epoch_loss_class += batch_loss_class.item()
            batch_loss_seg = beta * batch_loss_seg
            raw_loss_seg = loss_seg(pred_segm, mask) 
            batch_loss_seg = raw_loss_seg * weights.mean() * image.size(0)
            batch_loss = batch_loss_class + batch_loss_seg
            epoch_loss_seg += batch_loss_seg.item()
            
            
            pred_label = pred_class.argmax(dim=1)
            for i in range (len(dic_acc_disease) // 2):
                disease = reverse[i]
                idx_pred = (pred_label == i)
                idx_true = (label[:, 0] == i)
                
                dic_acc_disease[disease + "_tot"] += idx_true.sum().item()
                dic_acc_disease[disease + "_true"] += (idx_pred & idx_true).sum().item()
                

            # Backpropagation
            batch_loss.backward()
            optimizer.step()
            
        
        epoch_loss_class /= size
        epoch_loss_seg /= size
        epoch_loss = epoch_loss_class + epoch_loss_seg
        
        print(f'Epoch {epoch+1}/{num_epochs}, Total Loss: {epoch_loss},Classification Loss: {epoch_loss_class}, Segmentation Loss: {epoch_loss_seg}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')


'''

'\ndef train_loop_faster2(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama):\n\n    size = len(dataloader.dataset)  \n    model = model.to(device)\n    model.train()\n    \n    \n    for epoch in range(num_epochs):\n        epoch_loss_class = 0.0\n        epoch_loss_seg = 0.0\n        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}\n        for image, mask, label in dataloader:\n        \n            image= image.to(device)\n            mask= mask.to(device)\n            label= label.to(device)\n            \n            optimizer.zero_grad()\n        \n            pred_segm, pred_class = model(image)\n            \n            batch_loss_class = 0\n            batch_loss_seg = 0\n             \n            loss_c= loss_class(pred_class, label[:,0].long()) \n            loss_s = loss_seg(pred_segm, mask)  # unsqueeze\n            \n            penalty = (label[:, 2] == 0) 

Models 

In [ ]:

# segmentation :
model_segmentation = smp.Unet(
    encoder_name="resnet34",        
    encoder_weights="imagenet",     
    in_channels=3,                 
    classes=1,                      
)

# global classification  :
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    aux_params=dict(
        pooling='avg',             
        dropout=0.2,               
        classes=4, # A,D, G, N                 
    )
)



In [ ]:
model_segmentation2 = smp.Unet(
    encoder_name="resnet18",        
    encoder_weights="imagenet",     
    in_channels=3,                 
    classes=1,                      
)

# global classification  :
model2 = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    aux_params=dict(
        pooling='avg',             
        dropout=0.2,               
        classes=4, # A,D, G, N           
    )
)

In [ ]:
loss_seg= torch.nn.BCEWithLogitsLoss()  #############
loss_class = nn.CrossEntropyLoss()
loss_seg_faster= torch.nn.BCEWithLogitsLoss(reduction='none')
loss_class_faster = nn.CrossEntropyLoss(reduction='none')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)
num_epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
alpha = 1
beta = 1
gama = 2


### Train and test with train_loop_basic/test_loop_basic without data augmentation using Unet and resnet18

In [ ]:
imgs_epoch_basic_1 = train_loop_basic(train_dataloader, model2, loss_seg, loss_class, optimizer2, num_epochs, device, alpha, beta)

In [ ]:
all_pred, all_labels = test_loop_basic(train_dataloader, model2, loss_seg, loss_class, device, alpha, beta)

In [ ]:
imgs_per_epoch(imgs_epoch_basic_1)

##### Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_pred)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'D', 'G', 'N'])
cm_display.plot()
plt.plot()

### Train and test with train_loop_basic/test_loop_basic without data augmentation using Unet and resnet34

In [ ]:
imgs_epoch_basic_2 = train_loop_basic(train_dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta)

In [ ]:
all_pred2, all_labels2 = test_loop_basic(train_dataloader, model, loss_seg, loss_class, device, alpha, beta)

In [ ]:
imgs_per_epoch(imgs_epoch_basic_2)

#### Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels2, all_pred2)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'D', 'G', 'N'])
cm_display.plot()
plt.plot()

### Train and test with train_loop_faster/test_loop_faster without data augmentation using resnet18 and trying different parameters for alpha and beta

alpha = 1, beta = 1

In [ ]:
train_loop_faster(train_dataloader, model, loss_seg_faster, loss_class_faster, optimizer, num_epochs, device, alpha, beta, gama)

In [ ]:
all_pred3, all_labels3 = test_loop_faster(test_dataloader, model, loss_seg, loss_class, device, alpha, beta, gama)

#### Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels3, all_pred3)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'D', 'G', 'N'])
cm_display.plot()
plt.plot()

## Data augmentation

### Train and test with train_loop_faster/test_loop_faster with Unet and resnet34

In [ ]:
train_loop_faster(train_dataloader, model, loss_seg_faster, loss_class_faster, optimizer, num_epochs, device, alpha, beta, gama)

In [ ]:
all_pred5, all_labels5 = test_loop_faster(test_dataloader, model, loss_seg_faster, loss_class_faster, device, alpha, beta, gama)

#### Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels5, all_pred5)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'D', 'G', 'N'])
cm_display.plot()
plt.plot()

### Train and test with train_loop_faster/test_loop_faster with Unet and resnet18